# Optimization of food delivery by drones in Sierre's district

## Imports

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Tuple, Dict
from rich.console import Console
from IPython.display import clear_output
import json

from utils.data_loader import data_loader
from utils.utils import get_demand_mtx, cost_fct
from utils.solver import solve_cvrp
from utils.visualization import render_terminal_dashboard, plot_gantt_chart, plot_demand_curves

In [2]:
def simulation(distances_df: pd.DataFrame, altitudes_df: pd.DataFrame, commune_info_df: pd.DataFrame, hourly_weekly_demand_df: pd.DataFrame, battery_capacity : int = 4000,
              nb_drones: int = 30, scenario_id: int = 1, penalization: bool = False, penalty_per_unserved_order: float = 10.0, simulation_nb : str = "") -> Tuple[float, List[Dict]]:
    
    weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    
    scenarios = {
        1: {
            "name": "Full Operation (Extended Weekends)",
            "max_operating_hours": 100,
            "demand_threshold": 0.0,
            "schedule": {
                "Monday": list(range(10, 24)),    # 10:00 to 23:59
                "Tuesday": list(range(10, 24)),
                "Wednesday": list(range(10, 24)),
                "Thursday": list(range(10, 24)),
                "Friday": list(range(10, 25)),    # 10:00 to 00:59 (Includes hour 24)
                "Saturday": list(range(10, 25)),  # 10:00 to 00:59
                "Sunday": list(range(10, 24))
            }
        },
        2: {
            "name": "Peak Hours Only (Lunch & Dinner)",
            "max_operating_hours": 42,
            "demand_threshold": 0.0,
            # 12, 13 (Lunch) and 18, 19, 20, 21 (Dinner)
            "schedule": {day: [12, 13, 18, 19, 20, 21] for day in weekdays}
        }
    }

    if scenario_id not in scenarios:
        raise ValueError(f"Scenario {scenario_id} is not defined.")
        
    config = scenarios[scenario_id]
    active_schedule = config["schedule"]
    demand_threshold = config["demand_threshold"]
    maximum_operating_hours = config["max_operating_hours"]
    
    remaining_flying_time_per_drones = [0.0 for _ in range(nb_drones)]
    max_load = 12 
    # battery_capacity = 4000 
    
    total_energy_consumed = 0.0
    total_penalty_cost_chf = 0.0
    total_successful_deliveries = 0
    total_missed_deliveries = 0
    
    commune_names = commune_info_df.index.tolist()
    console = Console()
    simulation_history = []
    drone_activity_log = []
    
    # Use Monday, June 1, 2026 as the anchor for clean datetime math
    simulation_start = datetime(2026, 6, 1, 0, 0) 
    previous_time = None # Used to track real elapsed time between jumps

    console.print(f"[bold green]🚀 Starting Scenario {scenario_id}: {config['name']}[/bold green]")

    for day_idx, d in enumerate(weekdays):
        active_hours = active_schedule[d] # Get the specific hours for this day
        
        for h in active_hours: 
            for chunk in range(4): 
                
                # 1. Calculate Exact Current Time
                current_time = simulation_start + timedelta(days=day_idx, hours=h, minutes=chunk*15)
                
                # 2. Dynamic Cooldown tracking (Fixes the gap between 14:00 and 18:00)
                if previous_time is not None:
                    elapsed_minutes = (current_time - previous_time).total_seconds() / 60.0
                else:
                    elapsed_minutes = 15.0 # Default for the very first chunk
                    
                previous_time = current_time
                
                # Apply the elapsed time to the drones
                remaining_flying_time_per_drones = [max(0.0, t - elapsed_minutes) for t in remaining_flying_time_per_drones]
                available_drones = [True if t == 0.0 else False for t in remaining_flying_time_per_drones]
                
                # 3. Fetch Demand
                round_demand = get_demand_mtx(h, d, demand_threshold, hourly_weekly_demand_df, commune_info_df) 
                
                # 4. Solve 15-minute chunk
                routes, distance_per_drone, round_energy, remaining_flying_time_per_drones, route_times, dropped_orders, used_drone_ids = solve_cvrp(
                    available_vehicles=available_drones, 
                    round_demand=round_demand, 
                    remaining_flying_time_per_drones=remaining_flying_time_per_drones, 
                    max_load=max_load, 
                    battery_capacity=battery_capacity, 
                    penalization=penalization, 
                    penalty_per_unserved_order=penalty_per_unserved_order,
                    commune_names=commune_names,
                    distances_df=distances_df,
                    altitudes_df=altitudes_df
                )
                
                # 5. Process Data & Financials
                round_energy_wh = sum(round_energy)
                total_energy_consumed += round_energy_wh
                
                total_requested_this_round = int(np.sum(round_demand))
                served_this_round = total_requested_this_round - dropped_orders
                
                total_successful_deliveries += served_this_round
                total_missed_deliveries += dropped_orders
                
                if penalization and dropped_orders > 0:
                    total_penalty_cost_chf += (dropped_orders * penalty_per_unserved_order)
                
                # 6. JSON Snapshot Generation
                round_snapshot = {
                    "timestamp": current_time.isoformat(),
                    "day": d,
                    "hour_chunk": f"{h:02d}:{(chunk*15):02d}",
                    "financials": {
                        "round_energy_cost_chf": round_energy_wh * 0.00027,
                        "cumulative_energy_cost_chf": total_energy_consumed * 0.00027,
                        "penalties_incurred_chf": (dropped_orders * penalty_per_unserved_order) if penalization else 0.0
                    },
                    "metrics": {
                        "successful_deliveries": served_this_round,
                        "missed_deliveries": dropped_orders,
                        "cumulative_successful": total_successful_deliveries,
                        "cumulative_missed": total_missed_deliveries
                    },
                    "fleet_status": {
                        "drones": [{"id": i, "minutes_until_ready": t} for i, t in enumerate(remaining_flying_time_per_drones)]
                    },
                    "active_routes": []
                }

                if routes:
                    for idx, route in enumerate(routes):
                        round_snapshot["active_routes"].append({
                            "drone_id": used_drone_ids[idx],
                            "path": route, 
                            "energy_used_wh": round_energy[idx],
                            "distance_km": distance_per_drone[idx]
                        })
                        
                        # Add to Plotly Gantt Log
                        drone_activity_log.append({
                            "Drone": f"D{used_drone_ids[idx]:02d}",
                            "Start": current_time,
                            "Finish": current_time + timedelta(minutes=route_times[idx]),
                            "Status": "Flying",
                            "Path": " -> ".join(route)
                        })

                simulation_history.append(round_snapshot)

                # 7. Render live dashboard
                render_terminal_dashboard(current_time, round_energy_wh, total_energy_consumed, 
                                          routes, used_drone_ids, remaining_flying_time_per_drones, 
                                          dropped_orders, penalty_per_unserved_order, 
                                          total_successful_deliveries, total_missed_deliveries, console)

    final_cost = cost_fct(weekly_operating_minutes=maximum_operating_hours * 60, 
                          total_energy_consumed=total_energy_consumed, 
                          n_drones=nb_drones)
    
    final_cost += total_penalty_cost_chf 
    
    output_filename = f"simulation_results_scenario_{scenario_id}_{simulation_nb}.json"
    with open(output_filename, "w") as f:
        json.dump(simulation_history, f, indent=4)
        
    clear_output(wait=True)
    console.print(f"\n[bold green]✅ Simulation Complete. Final Weekly Cost (Incl. Penalties): CHF {final_cost:,.2f}[/bold green]")
    console.print(f"[bold blue]💾 Simulation state saved successfully to {output_filename}[/bold blue]")
    
    return final_cost, total_successful_deliveries, drone_activity_log

In [3]:
altitudes_df, distances_df, hourly_weekly_demand_df, commune_info_df = data_loader()
final_cost, total_successful_deliveries, drone_activity_log = simulation(distances_df, altitudes_df, commune_info_df, hourly_weekly_demand_df, scenario_id=1, battery_capacity=3000, simulation_nb="final_test")

✅ Simulation Complete. Final Weekly Cost (Incl. Penalties): CHF 41,509.39

💾 Simulation state saved successfully to simulation_results_scenario_1_final_test.json

In [4]:
final_cost, total_successful_deliveries, final_cost/total_successful_deliveries

(np.float64(41509.39481683229), 8376, np.float64(4.95575391795992))

In [5]:
plot_gantt_chart(drone_activity_log)

In [36]:
final_cost, total_successful_deliveries, drone_activity_log = simulation(distances_df, altitudes_df, commune_info_df, hourly_weekly_demand_df, scenario_id=2, battery_capacity=3000, simulation_nb="final_test")

✅ Simulation Complete. Final Weekly Cost (Incl. Penalties): CHF 18,527.89

💾 Simulation state saved successfully to simulation_results_scenario_2_final_test.json

In [38]:
final_cost, total_successful_deliveries, final_cost/total_successful_deliveries

(np.float64(18527.891589532344), 4840, np.float64(3.828076774696765))

In [ ]:
altitudes_df, distances_df, hourly_weekly_demand_df, commune_info_df = data_loader()
costs, n_deliveries = [], []
for i in range(10):
    final_cost, total_successful_deliveries, drone_activity_log = simulation(distances_df, altitudes_df, commune_info_df, hourly_weekly_demand_df, simulation_nb=str(i))
    costs.append(final_cost)
    n_deliveries.append(total_successful_deliveries)
    print(f"Simulation {i} - Final Cost: CHF {final_cost:,.2f}, Total Successful Deliveries: {total_successful_deliveries}, average cost per delivery: CHF {(final_cost/total_successful_deliveries) if total_successful_deliveries > 0 else float('inf'):.2f}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🕰️ Time: Monday 11:30               💸 Round Cost: CHF 1.7637              💰 Total Cost: CHF 6.6054            │
│ 📦 Total Delivered: 87              ❌ Total Missed: 0                     📊 Success Rate: 100.0%              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                        🚁 Active Routes this 15-Min Chunk                                         
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Drone ID   ┃ Flight Path                                                                                        ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Drone 04   │ Warehouse ➔ Grône ➔ Saint-Léonard ➔ Lens ➔ Lens ➔ Crans-Montana ➔ Crans-Montana ➔ Crans-Montana ➔  │
│            │ Crans-Montana ➔ Crans-Montana ➔ Warehouse                                                          │
│ Drone 06   │ Warehouse ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Sierre ➔ Noble-Contrée  │
│            │ ➔ Noble-Contrée ➔ Warehouse                                                                        │
│ Drone 07   │ Warehouse ➔ Chalais ➔ Anniviers ➔ Warehouse                                                        │
└────────────┴────────────────────────────────────────────────────────────────────────────────────────────────────┘

                                    ⏱️ Fleet Availability (Minutes until ready)                                    
┌──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ D00: 35.0m   │ D01: 78.0m   │ D02: 78.0m  │ D03: 48.0m  │ D04: 73.0m  │ D05: READY  │ D06: 72.0m  │ D07: 50.0m  │
│ D08: READY   │ D09: READY   │ D10: READY  │ D11: READY  │ D12: READY  │ D13: READY  │ D14: READY  │ D15: READY  │
│ D16: READY   │ D17: READY   │ D18: READY  │ D19: READY  │ D20: READY  │ D21: READY  │ D22: READY  │ D23: READY  │
│ D24: READY   │ D25: READY   │ D26: READY  │ D27: 83.0m  │ D28: READY  │ D29: READY  │             │             │
└──────────────┴──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘

Solving next chunk

In [23]:
import os
filename = "simulation_results_scenario_1_final_test.json"
def load_data(filepath):
    with open(filepath, "r") as f:
        return json.load(f)

history= load_data(os.path.join(filename))
print(history)

[{'timestamp': '2026-06-01T10:00:00', 'day': 'Monday', 'hour_chunk': '10:00', 'financials': {'round_energy_cost_chf': 0.11790139377171484, 'cumulative_energy_cost_chf': 0.11790139377171484, 'penalties_incurred_chf': 0.0}, 'metrics': {'successful_deliveries': 3, 'missed_deliveries': 0, 'cumulative_successful': 3, 'cumulative_missed': 0}, 'fleet_status': {'drones': [{'id': 0, 'minutes_until_ready': 0.0}, {'id': 1, 'minutes_until_ready': 0.0}, {'id': 2, 'minutes_until_ready': 0.0}, {'id': 3, 'minutes_until_ready': 0.0}, {'id': 4, 'minutes_until_ready': 0.0}, {'id': 5, 'minutes_until_ready': 0.0}, {'id': 6, 'minutes_until_ready': 0.0}, {'id': 7, 'minutes_until_ready': 0.0}, {'id': 8, 'minutes_until_ready': 0.0}, {'id': 9, 'minutes_until_ready': 0.0}, {'id': 10, 'minutes_until_ready': 0.0}, {'id': 11, 'minutes_until_ready': 0.0}, {'id': 12, 'minutes_until_ready': 0.0}, {'id': 13, 'minutes_until_ready': 0.0}, {'id': 14, 'minutes_until_ready': 0.0}, {'id': 15, 'minutes_until_ready': 0.0}, {'i

In [24]:
history = pd.read_json(filename)
history

,timestamp,day,hour_chunk,financials,metrics,fleet_status,active_routes
0,2026-06-01 10:00:00,Monday,10:00,"{'round_energy_cost_chf': 0.117901393771714, '...","{'successful_deliveries': 3, 'missed_deliverie...","{'drones': [{'id': 0, 'minutes_until_ready': 0...","[{'drone_id': 29, 'path': ['Warehouse', 'Sierr..."
1,2026-06-01 10:15:00,Monday,10:15,"{'round_energy_cost_chf': 0.13291245417171402,...","{'successful_deliveries': 3, 'missed_deliverie...","{'drones': [{'id': 0, 'minutes_until_ready': 0...","[{'drone_id': 28, 'path': ['Warehouse', 'Sierr..."
2,2026-06-01 10:30:00,Monday,10:30,"{'round_energy_cost_chf': 0.102889328971714, '...","{'successful_deliveries': 3, 'missed_deliverie...","{'drones': [{'id': 0, 'minutes_until_ready': 0...","[{'drone_id': 29, 'path': ['Warehouse', 'Sierr..."
3,2026-06-01 10:45:00,Monday,10:45,"{'round_energy_cost_chf': 0.13291306977171402,...","{'successful_deliveries': 3, 'missed_deliverie...","{'drones': [{'id': 0, 'minutes_until_ready': 0...","[{'drone_id': 28, 'path': ['Warehouse', 'Sierr..."
4,2026-06-01 11:00:00,Monday,11:00,"{'round_energy_cost_chf': 1.713369267986692, '...","{'successful_deliveries': 25, 'missed_deliveri...","{'drones': [{'id': 0, 'minutes_until_ready': 5...","[{'drone_id': 0, 'path': ['Warehouse', 'Chalai..."
...,...,...,...,...,...,...,...
395,2026-06-07 22:45:00,Sunday,22:45,"{'round_energy_cost_chf': 0.302882764771411, '...","{'successful_deliveries': 6, 'missed_deliverie...","{'drones': [{'id': 0, 'minutes_until_ready': 0...","[{'drone_id': 28, 'path': ['Warehouse', 'Sierr..."
396,2026-06-07 23:00:00,Sunday,23:00,"{'round_energy_cost_chf': 0.0, 'cumulative_ene...","{'successful_deliveries': 1, 'missed_deliverie...","{'drones': [{'id': 0, 'minutes_until_ready': 0...",[]
397,2026-06-07 23:15:00,Sunday,23:15,"{'round_energy_cost_chf': 0.0, 'cumulative_ene...","{'successful_deliveries': 1, 'missed_deliverie...","{'drones': [{'id': 0, 'minutes_until_ready': 0...",[]
398,2026-06-07 23:30:00,Sunday,23:30,"{'round_energy_cost_chf': 0.0, 'cumulative_ene...","{'successful_deliveries': 1, 'missed_deliverie...","{'drones': [{'id': 0, 'minutes_until_ready': 0...",[]


In [34]:
history.active_routes.iloc[15][0]["distance_km"]

33.24165923004841

In [35]:
cumulative_energy_cost = history.financials.iloc[-1]["cumulative_energy_cost_chf"]

total_km, n_flies = 0, 0
for i in range(history.shape[0]):
    for drone in history.active_routes.iloc[i]:
        total_km += drone["distance_km"]
        n_flies += 1
           
total_km, n_flies, cumulative_energy_cost

(29876.314247606744, 952, 528.3948168322919)

In [37]:
filename = "simulation_results_scenario_2_final_test.json"
history = pd.read_json(filename)

cumulative_energy_cost = history.financials.iloc[-1]["cumulative_energy_cost_chf"]

total_km, n_flies = 0, 0
for i in range(history.shape[0]):
    for drone in history.active_routes.iloc[i]:
        total_km += drone["distance_km"]
        n_flies += 1
           
total_km, n_flies, cumulative_energy_cost

(17401.52794920662, 527, 311.8915895323443)